In [ ]:
import sys
sys.path.append('../')

from src.run_app import *
from src.prepare_gam import *
from src.utils import *
from gam_rs_utils.binarize_dataset import binarize_dataset
from gam_rs_utils.utils import *
from FasterRisk.src.fasterrisk import fasterrisk

import pickle
import numpy as np
from time import time

def rashomon_set_size(X_one_hot, y, swaps, gt, params, use_beam):
    rs = fasterrisk.RiskScoreOptimizer(X_one_hot, y, k=10, lb=-100, ub=100, gap_tolerance=gt, select_top_m=-1, maxAttempts=25)
    if use_beam:
        rs.optimize_with_swaps_beam_search(swaps=swaps, **params)
    else:
        rs.optimize_with_swaps(swaps=swaps, **params)
    return rs.sparseDiversePool_betas, rs.sparseDiversePool_beta0

def bisect_left_one(func, low, high, tol=1e-6):
    """Find leftmost such that func(x) >= 1"""
    record = []
    while high - low > tol:
        mid = (low + high) / 2
        start = time()
        betas, beta0 = func(mid)
        end = time()
        num_solutions = betas.shape[0]
        print(f"\ttried {mid}, got {num_solutions} solutions, took {end - start:.2f} seconds")
        record.append((mid, end - start, betas.copy(), beta0.copy()))
        if num_solutions < 1:
            low = mid
        else:
            high = mid
    return record

dataset_settings = {
    "bank": {
        "left": 0.0005, 
        "right": 0.001, 
        "num_estimators": 100
    },
    "compas": {
        "left": 0.0001,
        "right": 0.002,
        "num_estimators": 100
    },
    "diabetes": {
        "left": 0.0,
        "right": 0.005, 
        "num_estimators": 500
    },
    # "heloc_original": {"left": 0.0001, "right": 0.001, "num_estimators": 50},
    # "hiv": {"left": 0.0001, "right": 0.001, "num_estimators": 50},
    # "netherlands": {"left": 0.0001, "right": 0.001, "num_estimators": 100},
    "spambase": {
        "left": 0.0001, 
        "right": 0.001, 
        "num_estimators": 100
    },
}

def run_swap_tests(use_beam, params, file_name):
    swaps = 5
    results = []
    for dataset_name, settings in dataset_settings.items():
        left, right, ne = settings["left"], settings["right"], settings["num_estimators"]

        path = '../datasets/{}.csv'.format(dataset_name)
        dataset = pd.read_csv(path)
        print(f"Dataset: {dataset_name}")
        print(f"Binarized shape: {dataset.shape}")

        df, thresholds, header, threshold_guess_time = binarize_dataset(dataset, ne)
        X, y = df.iloc[:, :-1], df.iloc[:, -1]
        header = pd.Index(["intercept"] + list(X.columns)).astype("object")
        X_one_hot, y = utils.get_X_y(X, y)

        for s in range(1, swaps + 1):
            print(f"Swapping {s} times")
            gam_results = bisect_left_one(lambda gt: rashomon_set_size(X_one_hot, y, s, gt, params, use_beam), left, right)
            for gr in gam_results:
                result = {
                    "dataset": dataset_name,
                    "dataset_shape": dataset.shape,
                    "num_estimators": ne,
                    "gap_tolerance": gr[0],
                    "feature_selection": "top",
                    "swaps": s,
                    "threshold_guess_time": threshold_guess_time,
                    "num_features": len(header),
                    "runtime": gr[1],
                    "betas": gr[2],
                    "beta0": gr[3],
                    "num_solutions": gr[2].shape[0],
                    "loss": get_loss(X_one_hot, y, gr[3], gr[2])
                }
                results.append(result)

    with open(f"results/{file_name}.pkl", "wb") as f:
        pickle.dump(results, f)

In [5]:
run_swap_tests(
    False,
    {"fanout_decay": 1, "feature_selection": "top"},
    "swaps"
)

Dataset: bank
Binarized shape: (4521, 17)
Swapping 1 times
	tried 0.00075, got 2 solutions, took 16.96 seconds
	tried 0.000625, got 1 solutions, took 12.18 seconds
	tried 0.0005625000000000001, got 1 solutions, took 16.56 seconds
	tried 0.00053125, got 0 solutions, took 12.01 seconds
	tried 0.000546875, got 0 solutions, took 13.33 seconds
	tried 0.0005546875000000001, got 1 solutions, took 18.33 seconds
	tried 0.0005507812500000001, got 0 solutions, took 12.33 seconds
	tried 0.0005527343750000001, got 1 solutions, took 16.91 seconds
	tried 0.0005517578125000001, got 0 solutions, took 18.86 seconds
Swapping 2 times
	tried 0.00075, got 0 solutions, took 15.56 seconds
	tried 0.000875, got 29 solutions, took 13.95 seconds
	tried 0.0008125000000000001, got 0 solutions, took 17.71 seconds
	tried 0.0008437500000000001, got 29 solutions, took 20.52 seconds
	tried 0.0008281250000000001, got 28 solutions, took 17.86 seconds
	tried 0.0008203125000000001, got 0 solutions, took 13.48 seconds
	tried

In [2]:
run_swap_tests(
    True,
    {"beam_size": 200},
    "swaps_beam"
)

Dataset: bank
Binarized shape: (4521, 17)
Swapping 1 times
	tried 0.00075, got 2 solutions, took 9.74 seconds
	tried 0.000625, got 1 solutions, took 10.34 seconds
	tried 0.0005625000000000001, got 1 solutions, took 9.32 seconds
	tried 0.00053125, got 0 solutions, took 9.74 seconds
	tried 0.000546875, got 0 solutions, took 10.72 seconds
	tried 0.0005546875000000001, got 1 solutions, took 9.66 seconds
	tried 0.0005507812500000001, got 0 solutions, took 9.25 seconds
	tried 0.0005527343750000001, got 1 solutions, took 10.96 seconds
	tried 0.0005517578125000001, got 0 solutions, took 10.79 seconds
Swapping 2 times
	tried 0.00075, got 0 solutions, took 10.65 seconds
	tried 0.000875, got 29 solutions, took 10.55 seconds
	tried 0.0008125000000000001, got 0 solutions, took 9.47 seconds
	tried 0.0008437500000000001, got 29 solutions, took 10.85 seconds
	tried 0.0008281250000000001, got 28 solutions, took 11.89 seconds
	tried 0.0008203125000000001, got 0 solutions, took 10.61 seconds
	tried 0.000